# B2.11 · White-box agentic pentest — the source, and what it lets you prove

**Function B — Application Security with an AI SDLC → The AI SDLC — an Agentic AppSec Pipeline, Before and After Deploy**

Builds on **[B2.10 · Agentic penetration testing — the loop, and who runs each turn](https://spbreed.github.io/cyber-commons/lessons/B2.10.html)**.

| | |
|---|---|
| Tools used | Semgrep |

## What this lesson is

**What it covers.** A white-box engagement: enumerating real paths from entry point to sink, naming the authorisation predicate on each, and separating reachable sinks from present ones.

**Why a security engineer needs it.** Full source produces the most useless report in security — every sink that exists, with no statement about reachability or entitlement. Two distinctions fix it, and the second, that authentication is not authorisation, is the shape of every BOLA finding.

| | |
|---|---|
| **Day 0 — why** | Full source produces a finding list nobody can act on, because presence is reported where reachability was needed. |
| **Day 1 — how** | Enumerate paths from real entry points to sinks, and classify the predicate on each hop as authentication or authorisation. |
| **Day 2 — measure** | Sinks reachable from an entry point, and how many of those carry only an authentication check. Three of five, on CyberTravels. |

## 1 · The hook

You were handed the whole repository, and the report is a list of four hundred sinks nobody will read. The two facts that turn it into findings are which sinks a request can actually reach, and which of those check that the caller owns the thing — not merely that they are logged in.

> **At CyberTravels.** The tree is CyberTravels', and the path that matters is the refund one: handler to svc.refund to the payments sink, authenticated by a session at every hop and checked for ownership at none. That is the refund incident stated as a finding before it happened.

## 2 · The framework

```
   a sink list                     paths, with the predicate on each

   admin.reset_all      present    POST /refunds
   payments.refund      present      handler [authn: session]
   db.bookings.read     present        svc.refund [authz: NONE]
   ...395 more...                        -> payments.refund   <- FINDING

   presence is a scanner dump.        session proves you are SOMEBODY.
   reachability + who-is-allowed      owner==caller proves you may touch
   is a finding.                      THIS object. only the second is authz.
```

White box is the engagement where you are handed everything: the repository at a
pinned commit, the dependency lock, the IaC plan, the tool and MCP manifests,
the IAM policy, the API schema. An agent with all of that can enumerate more in
an afternoon than a black-box tester finds in a week.

It can also produce the least useful report in security, and usually does: a
list of every sink that exists, with no statement about whether anything reaches
them or who is allowed to. That list is unactionable in exactly the way a
scanner dump is, which wastes the access.

Two distinctions turn the dump into findings, and both are cheap once the agent
has drawn the paths rather than the sinks.

**Reachability.** A sink with no path from any entry point is a different object
from one with three. Report the unreachable ones *as* unreachable — they are one
feature away from being findings, and the feature that adds the route will not
re-run this analysis.

**Authentication is not authorisation.** `session` proves the caller is
somebody. `owner == caller` proves they are entitled to *this* object. A
reachable path carrying only the first kind is the shape of every BOLA finding,
and it is the shape of the CyberTravels refund incident — the caller was
authenticated from end to end.

The data sources are the point of the mode: name which one establishes each
fact, and the finding is checkable rather than asserted.

## 3 · The procedure, as a skill

The skill enumerates paths from four CyberTravels entry points to their sinks, classifies the predicate on each hop as authentication or authorisation, and separates reachable sinks from ones that are merely present.

### The skill — [`skills/redteam/whitebox-path-reachability/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/redteam/whitebox-path-reachability/SKILL.md)

```yaml
name: whitebox-path-reachability
description: >-
  With source in hand, enumerate paths from real entry points to sinks and name
  the authorisation predicate on each, separating reachable sinks from merely
  present ones. Use for a white-box engagement, or when a scanner has produced a
  finding list nobody can act on.
allowed-tools: Read, Grep, Glob
```

# Presence is not reachability, and authentication is not authorisation

White box is the mode where you are handed everything, and the failure it
produces is specific: a long list of sinks that exist, with no statement about
whether anything can reach them or who is allowed to. That list is unactionable
in exactly the way a scanner's output is, which is a waste of the access you
were given.

Two distinctions do the work, and both are cheap once the paths are drawn.

## When to use this

A white-box engagement, a threat model that needs to be argued rather than
asserted, and any time a finding list is long and nobody is fixing it.

## Procedure

**1 — Write down what you were handed, and what each source establishes.**
Repository at a pinned commit, dependency lock, IaC plan, tool and MCP
manifests, IAM policy, API schema, prior findings. Naming the source per fact is
the difference between a finding and an assertion.

**2 — Enumerate paths, not sinks.** Entry point, every hop, the sink. A sink
with no path from any entry point is a different kind of object from one with
three.

**3 — Record the predicate on each hop, and classify it.** `session` and
`service_account` prove the caller is *somebody*. `owner==caller` and role
checks prove they are entitled to *this object*. A path carrying only the first
kind is the shape every BOLA finding has.

**4 — Report the unreachable sinks as unreachable, not as nothing.** They are
one route away from being findings, and the feature that adds the route will not
re-run this analysis on its own.

## Example

```
  AUTHN ONLY POST /refunds               handler -> svc.refund -> payments.refund
            authn: session   authz: NONE ON ANY HOP

sinks present in the tree      7
sinks reachable from an entry  5
  of those, authenticated only 3
```

The run continues past this. The script is the example: `test_skills.py`
executes it on every build, so this block cannot drift from what the skill
actually prints.

## Output contract

```json
{
  "data_sources": [{"source": "str", "establishes": "str"}],
  "paths": [{"entry": "str", "hops": [["str", "str|null"]], "sink": "str",
             "reachable": true, "authn": ["str"], "authz": ["str"]}],
  "counts": {"present": 0, "reachable": 0, "authn_only": 0, "unreachable": 0},
  "findings": [{"sink": "str", "entry": "str", "has": ["str"]}]
}
```

## Failure modes

- **Counting a session check as authorisation.** It is the single most common
  reason a BOLA survives a white-box engagement.
- **Dropping unreachable sinks.** They are the next release's findings.
- **Reporting sinks instead of paths.** A sink reachable three ways needs three
  fixes or one chokepoint, and the list does not say which.
- **Trusting the manifest over the code.** The manifest says what an agent may
  call; the call graph says what it does.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/redteam/whitebox-path-reachability/scripts/whitebox_path_reachability.py
SCRIPT = "skills/redteam/whitebox-path-reachability/scripts/whitebox_path_reachability.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.5 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Of seven sinks in the tree, five are reachable from an entry point and two are reported as present-but-unreachable rather than dropped. Three of the reachable five carry only a session or service-account check — authentication, not authorisation — including the refund path, which is the incident's shape stated before the incident.

## Your turn

Take one endpoint in your own estate and trace it to its sink by hand. Count the hops that check the caller is somebody against the hops that check they own this object. If the second count is zero, you have found your first finding.

---

**Next → [B2.12 · Black-box agentic pentest — inference, and refusing to report it as fact](https://spbreed.github.io/cyber-commons/lessons/B2.12.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.11.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.11.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*